In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
from grid_generator import generate_grid
from heuristics import heuristics
from astar import astar

In [ ]:
NUM_GRIDS = 20
GRID_ROWS = 20
GRID_COLS = 20
OBSTACLE_PROB = 0.25

print(f"Number of grids: {NUM_GRIDS}")
print(f"Grid dimensions: {GRID_ROWS} x {GRID_COLS}")
print(f"Obstacle probability: {OBSTACLE_PROB}")

In [ ]:
grid_dataset = []

for i in range(NUM_GRIDS):

    grid, start, goal = generate_grid(
        rows=GRID_ROWS,
        cols=GRID_COLS,
        obstacle_prob=OBSTACLE_PROB,
        ensure_path=True
    )

    grid_dataset.append({
        "grid": grid,
        "start": start,
        "goal": goal
    })

print(f"Successfully generated {len(grid_dataset)} grids.")

In [ ]:
results = []

for grid_id, data in enumerate(grid_dataset, start=1):

    grid = data["grid"]
    start = data["start"]
    goal = data["goal"]

    for heuristic_name, heuristic_func in heuristics.items():

        result = astar(
            grid,
            start,
            goal,
            heuristic_func
        )

        results.append({
            "Grid": grid_id,
            "Heuristic": heuristic_name,
            "Success": result["success"],
            "Path Length": result["path_length"],
            "Nodes Expanded": result["nodes_expanded"],
            "Execution Time (ms)": result["execution_time"] * 1000,
            "Max Frontier Size": result["max_frontier_size"]
        })

print("All experiments completed!")

In [ ]:
df = pd.DataFrame(results)

df.head(10)

In [ ]:
summary = df.groupby("Heuristic").agg({
    "Execution Time (ms)": ["mean", "std"],
    "Nodes Expanded": ["mean", "std"],
    "Path Length": ["mean", "std"],
    "Max Frontier Size": ["mean", "std"],
    "Success": "mean"
}).round(3)

summary

In [ ]:
plt.figure(figsize=(10, 6))

sns.lineplot(
    data=df,
    x="Grid",
    y="Execution Time (ms)",
    hue="Heuristic",
    marker="o",
    linewidth=2
)

plt.title("A* Heuristic Comparison - Execution Time")
plt.xlabel("Grid Number")
plt.ylabel("Execution Time (ms)")
plt.xticks(range(1, NUM_GRIDS + 1))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.lineplot(
    data=df,
    x="Grid",
    y="Nodes Expanded",
    hue="Heuristic",
    marker="o",
    linewidth=2
)

plt.title("A* Heuristic Comparison - Nodes Expanded")
plt.xlabel("Grid Number")
plt.ylabel("Number of Nodes Expanded")
plt.xticks(range(1, NUM_GRIDS + 1))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.lineplot(
    data=df,
    x="Grid",
    y="Path Length",
    hue="Heuristic",
    marker="o",
    linewidth=2
)

plt.title("A* Heuristic Comparison - Path Length")
plt.xlabel("Grid Number")
plt.ylabel("Path Length (Number of Moves)")
plt.xticks(range(1, NUM_GRIDS + 1))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.lineplot(
    data=df,
    x="Grid",
    y="Max Frontier Size",
    hue="Heuristic",
    marker="o",
    linewidth=2
)

plt.title("A* Heuristic Comparison - Maximum Frontier Size")
plt.xlabel("Grid Number")
plt.ylabel("Maximum Frontier Size")
plt.xticks(range(1, NUM_GRIDS + 1))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
metrics = [
    "Execution Time (ms)",
    "Nodes Expanded",
    "Max Frontier Size"
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, metrics):

    sns.barplot(
        data=df,
        x="Heuristic",
        y=metric,
        estimator="mean",
        errorbar="sd",
        ax=ax
    )

    ax.set_title(f"Average {metric}")
    ax.set_xlabel("Heuristic")
    ax.set_ylabel(metric)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle(
    "Average A* Performance Across 20 Random Grids",
    fontsize=15
)

plt.tight_layout()
plt.show()